# Description for the test case
Flat plate suddenly accelerated inside a static laminar newtonian incompresible ideal flow with constant propierties

## Goberning equations
$$ \nabla \vec{u}=0$$
$$ \frac{\partial \vec{u}}{\partial t} + (\vec{u}\nabla \vec{u}) = -\nabla p + \frac{1}{Re} \nabla^2\vec{u} $$
$$ \frac{\partial T}{\partial t} + (\vec{u}\nabla T) = \frac{1}{RePr} \nabla^2T \frac{Ec}{Re}\phi $$
$$ \phi = 2\left[ \left( \frac{\partial u}{\partial x}\right)^2 + \left( \frac{\partial v}{\partial y}\right)^2\right] + \left[ \frac{\partial u}{\partial y} + \frac{\partial v}{\partial x}\right]^2 $$

## Finite difference simulation:

Estructure PISO: (first we resolve the velocity components evoulution neglecting the pressure effects, and later we correct them)

$$ u_{i,j}^* = u_{i,j}^n + \Delta t \left[ -\left( u\partial_x u + v \partial_y u \right)_{i,j} + \frac{1}{Re} \nabla^2u_{i,j} \right]$$
$$ v_{i,j}^* = v_{i,j}^n + \Delta t \left[ -\left( u\partial_x v + v \partial_y v \right)_{i,j} + \frac{1}{Re} \nabla^2v_{i,j} \right]$$

Poisson equation for pressure:

$$ \nabla^2p_{i,j}^{n+1} = \frac{1}{\Delta t} \left[ \frac{u_{i,j+1}^{n+1} - u_{i,j-1}^n}{2\Delta x} + \frac{v_{i,j+1}^{n+1} - v_{i,j-1}^n}{2\Delta y} \right]$$

$$ u_{i,j}^{n+1} = u_{i,j}^* - \Delta t \frac{p_{i,j+1}^{n+1} - p_{i,j-1}^{n+1}}{2\Delta x} $$
$$ v_{i,j}^{n+1} = v_{i,j}^* - \Delta t \frac{p_{i+1,j}^{n+1} - p_{i-1,j}^{n+1}}{2\Delta x} $$

Energy equation:

$$ T_{i,j}^{n+1} = T_{i,j}^n + \Delta t \left[ -\left( u\partial_x T + v \partial_y T \right)_{i,j} + \frac{1}{RePr} \nabla^2 T_{i,j} + \frac{Ec}{Re} \phi_{i,j}\right]$$


Boundary conditions:

$$ \vec{u}|_{y=0,L}=0 $$
$$ u|_{x=-2} = u|_{x=-1} $$
$$ p|_{x=0} = p|_{x=1} $$
$$ p|_{y=0} = p|_{y=1} $$
$$ p|_{x=-2} = p|_{x=-1} $$

## Aditional surmises:

No slip condition in the surface of the plate
Gravitational effects neglected


In [ ]:
import sys
import matplotlib.pyplot as plt
import numpy as np
import os
import pickle
from numba import njit, prange
from matplotlib.animation import FuncAnimation, FFMpegWriter, PillowWriter
from IPython.display import display, HTML
from scipy.interpolate import RegularGridInterpolator
import time
import pandas as pd
from copy import deepcopy
import seaborn as sns
from numba import set_num_threads
from itertools import product
from tqdm import tqdm
from collections import defaultdict
import copy

plt.rcParams.update({
    "text.usetex": True,
    "font.family": "serif",
    "font.size": 20,
    "axes.labelsize": 20,
    "axes.titlesize": 20,
    "xtick.labelsize": 20,
    "ytick.labelsize": 20,
    "legend.fontsize": 20,
})

set_num_threads(14)  # Ajusta al nº de núcleos que quieras